# SoloFinanzas · Benchmark de categorización

Este notebook muestra el piloto de **modelos gratuitos**. En Jupyter local lee los resultados y la hoja de revisión desde `backend/data/benchmark/pilot/`. En Google Colab funciona con el archivo agregado `summary_provisional.json`.

**Privacidad:** el notebook versionado no contiene movimientos, descripciones, claves ni resultados individuales. Para Colab sube solamente `summary_provisional.json` desde `backend/data/benchmark/pilot/`. Los detalles por movimiento se muestran únicamente en Jupyter local.

## 1. Cargar datos

En Jupyter local selecciona el entorno Python del backend. Si faltan librerías visuales, instala `experiments/categorization/requirements-notebook.txt` desde `backend/`. En Colab las librerías suelen estar disponibles; sube manualmente el resumen agregado al panel de archivos.

In [ ]:
from pathlib import Path
import csv
import json
import os
import sys
import tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "solofinanzas-mpl"))

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display, Markdown
except ImportError as exc:
    raise RuntimeError(
        "Instala las dependencias desde backend/: "
        "python -m pip install -r experiments/categorization/requirements-notebook.txt"
    ) from exc

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
def find_local_dataset():
    for root in (Path.cwd(), *Path.cwd().parents):
        for candidate in (
            root / "backend" / "data" / "benchmark" / "pilot",
            root / "data" / "benchmark" / "pilot",
        ):
            if (candidate / "review.csv").is_file():
                return candidate.resolve()
    return None

DATA_DIR = find_local_dataset()
if DATA_DIR:
    BACKEND_DIR = DATA_DIR.parents[2]
    sys.path.insert(0, str(BACKEND_DIR))
    from experiments.categorization.benchmark import report
    try:
        common = report(DATA_DIR)
        local_full = report(DATA_DIR, models={"rules_current", "laya_B", "mdeberta_B"})
        label_status = "verificadas"
    except ValueError:
        common = report(DATA_DIR, provisional=True)
        local_full = report(
            DATA_DIR, provisional=True,
            models={"rules_current", "laya_B", "mdeberta_B"},
        )
        label_status = "históricas sin verificar"
    data = {"all_models_common_cases": common, "local_full_sample": local_full}
    display(Markdown(f"**Modo local.** Datos: `{DATA_DIR}`. Etiquetas: **{label_status}**."))
else:
    options = (Path.cwd() / "summary_provisional.json", Path("/content/summary_provisional.json"))
    summary_path = next((path for path in options if path.is_file()), None)
    if summary_path is None:
        raise FileNotFoundError(
            "No encuentro la base local ni summary_provisional.json. "
            "En Colab, sube solo el archivo agregado al panel de archivos."
        )
    data = json.loads(summary_path.read_text(encoding="utf-8"))
    label_status = "históricas sin verificar"
    display(Markdown(f"**Modo agregado.** Archivo: `{summary_path.name}`. Sin datos individuales."))

## 2. Cobertura y aciertos

Las barras de aciertos usan como denominador **todos los casos**, incluidos los enviados a revisión. Así se ve el valor práctico de cada modelo. Las etiquetas del piloto siguen siendo provisionales hasta que se revise `review.csv`.

In [ ]:
common_df = pd.DataFrame(data["all_models_common_cases"])
local_df = pd.DataFrame(data["local_full_sample"])
def plot_coverage(frame, title):
    frame = frame.sort_values("correct_per_case")
    names = frame["model"].str.replace("_", " ", regex=False)
    y = range(len(frame))
    fig, ax = plt.subplots(figsize=(10, max(3.4, 0.55 * len(frame) + 1)))
    ax.barh(y, frame["coverage"] * 100, color="#a8c9c3", label="Respondió")
    ax.barh(y, frame["correct_per_case"] * 100, color="#16665a", label="Coincidió con etiqueta")
    ax.set_yticks(list(y), names)
    ax.set_xlim(0, 100)
    ax.set_xlabel("% de todos los casos")
    ax.set_title(f"{title} · {int(frame['evaluated'].iloc[0])} casos")
    ax.legend(loc="lower right")
    ax.grid(axis="x", alpha=0.2)
    display(fig)
    plt.close(fig)

plot_coverage(common_df, "Comparación común")
plot_coverage(local_df, "Muestra local")

## 3. Casos sin regla y decisiones de revisión

La comparación común se seleccionó entre movimientos donde las reglas actuales no proponían categoría. La columna «aciertos» se basa en la etiqueta histórica. Un `needs_review` evita asignar una categoría sin evidencia suficiente.

In [ ]:
comparison_cols = [
    "model", "evaluated", "rule_miss_answered", "rule_miss_correct",
    "needs_review_count", "auto_90_count", "auto_90_correct",
    "transfer_confusions",
]
display(
    common_df[comparison_cols]
    .rename(columns={
        "model": "Modelo",
        "evaluated": "Casos",
        "rule_miss_answered": "Respondió",
        "rule_miss_correct": "Coincidió",
        "needs_review_count": "A revisión",
        "auto_90_count": "Prob. ≥ 0,90",
        "auto_90_correct": "Coincidió ≥ 0,90",
        "transfer_confusions": "Confusiones de transferencia",
    })
    .sort_values("Coincidió", ascending=False)
    .reset_index(drop=True)
)

## 4. Latencia

Tiempo mediano por movimiento en este equipo o a través de la API. La inferencia local no tiene costo por llamada, pero utiliza CPU y almacenamiento. Mercury Decide gratuito consumió 31 solicitudes y OpenRouter registró USD 0 en este piloto.

In [ ]:
latency = common_df[common_df["model"] != "rules_current"].sort_values("p50_latency_ms")
fig, ax = plt.subplots(figsize=(9, max(3.2, 0.55 * len(latency) + 1)))
bars = ax.barh(latency["model"], latency["p50_latency_ms"], color="#327f91")
ax.set_xlabel("Milisegundos, mediana")
ax.set_title("Latencia por movimiento · muestra común")
ax.grid(axis="x", alpha=0.2)
for bar, ms in zip(bars, latency["p50_latency_ms"]):
    ax.text(bar.get_width() + 40, bar.get_y() + bar.get_height()/2, f"{ms:.0f} ms", va="center")
ax.set_xlim(0, latency["p50_latency_ms"].max() * 1.3)
display(fig)
plt.close(fig)

## 5. Revisar los 20 movimientos del piloto externo

Esta tabla aparece solo en Jupyter local. Muestra el **texto depurado** y las predicciones; no muestra la descripción original. Para corregir las etiquetas, edita `review.csv` y usa `taxonomy.csv` como referencia de IDs. Marca `verified=si` en cada fila revisada y vuelve a ejecutar este notebook.

In [ ]:
if DATA_DIR:
    with (DATA_DIR / "review.csv").open(encoding="utf-8-sig", newline="") as handle:
        review_rows = list(csv.DictReader(handle))
    taxonomy = {
        row["id"]: row["path"]
        for row in (json.loads(line) for line in
                    (DATA_DIR / "taxonomy.jsonl").read_text(encoding="utf-8").splitlines())
    }
    def predictions(name):
        path = DATA_DIR / f"predictions_{name}.jsonl"
        if not path.is_file():
            return {}
        return {row["case_id"]: row for row in
                (json.loads(line) for line in path.read_text(encoding="utf-8").splitlines())}
    pred = {name: predictions(name) for name in ("mercury_B", "laya_B", "mdeberta_B")}
    def label_for(name, case_id):
        item = pred[name].get(case_id)
        if not item:
            return "—"
        if item["category_id"] is None:
            return "Revisión" if item.get("abstained") else "Sin respuesta"
        return taxonomy.get(item["category_id"], "ID desconocido")
    detail = []
    for row in review_rows:
        if row["external_approved"].strip().lower() not in {"si", "sí", "yes", "1"}:
            continue
        case_id = row["case_id"]
        detail.append({
            "Descripción depurada": row["description_depurada_a_enviar"],
            "Etiqueta histórica": row["historical_category_path"],
            "Mercury B": label_for("mercury_B", case_id),
            "Laya B": label_for("laya_B", case_id),
            "mDeBERTa B": label_for("mdeberta_B", case_id),
            "Verificada": row["verified"] or "pendiente",
        })
    display(pd.DataFrame(detail))
else:
    display(Markdown("Los detalles permanecen en el equipo local; Colab muestra solo cifras agregadas."))

## Interpretación

- El piloto externo contiene 20 casos difíciles seleccionados porque las reglas no daban categoría; no representa por sí solo todo el tráfico.
- Las probabilidades de modelos distintos no están calibradas entre sí. Los umbrales 0,70 y 0,90 son exploratorios.
- Después de verificar las etiquetas, compara de nuevo cobertura, aciertos y errores graves. Una sola muestra pequeña no basta para automatizar movimientos financieros.
- Código y metodología: [README del experimento](README.md).